# 第2章 画像と動画を言葉にする (Cosmos 3 Edge の Reasoner)

Cosmos 3 Edge の **Reasoner** は、画像や動画を見て文章で答える部分です。この章では次のことを順に試します。

1. 画像を細かく説明させる (キャプション)
2. 画像について質問する (VQA)。日本語で答えさせる
3. 動画を説明させる
4. 「考えるモード」で、見えていることから推論させる (常識推論)
5. 今の状況と、次に起きることを予想させる
6. ロボットのプログラムで使いやすいように、見えているものを JSON で出させる
7. 自分の画像で試す

素材は NVIDIA の公式クックブックとモデルカードのお手本を使います。

## 準備

**ランタイム**: 「ランタイム → ランタイムのタイプを変更」で **L4** か **A100** の GPU を選んでください (T4 は対象外。Ampere 以降の GPU と bf16 が前提です)。
Reasoner (文章で答える部分) だけを使うので、GPU メモリは 10GB 前後で足ります。

上から順に実行してください。「モデルを読み込む」セルは初回に 5GB ほどダウンロードします。

In [ ]:
# @title GPU を確かめる
import torch
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0), f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.0f} GB")
    assert torch.cuda.get_device_capability()[0] >= 8, "Ampere 以降の GPU (L4 / A100 など) を選んでください。T4 では動きません。"
else:
    print("GPU が見つかりません。Colab ならランタイムのタイプで L4 か A100 を選んでください。")

In [ ]:
# @title インストール (1〜2 分)
!pip install -q -U "transformers>=5.18" accelerate av huggingface_hub

In [ ]:
# @title サンプル素材を取ってくる関数
import os
import urllib.request
from pathlib import Path

ASSETS = Path(os.environ.get("COSMOS_ASSETS", "assets"))
ASSETS.mkdir(parents=True, exist_ok=True)
# NVIDIA の公式クックブック (cosmos-framework) とモデルカードに置かれているお手本の素材
FW = "https://media.githubusercontent.com/media/NVIDIA/cosmos-framework/main/.agents/skills/cookbooks/cosmos3"
HF = "https://huggingface.co/nvidia/Cosmos3-Edge/resolve/main/assets"


def fetch(url):
    """URL の素材を assets/ に落として、そのパスを返す (2 回目からは落とさない)"""
    path = ASSETS / url.rsplit("/", 1)[-1]
    if not path.exists():
        urllib.request.urlretrieve(url, path)
    return str(path)

In [ ]:
# @title モデルを読み込む (初回は 5GB ほどダウンロード)
import json
import re

from IPython.display import Image as IPImage, Video, display
from PIL import Image, ImageDraw, ImageOps
from transformers import AutoModelForImageTextToText, AutoProcessor

MODEL_ID = "nvidia/Cosmos3-Edge"
# Transformers は統合チェックポイントから Reasoner の部分だけを読み込む (生成用の部分は捨てる)
model = AutoModelForImageTextToText.from_pretrained(MODEL_ID, dtype=torch.bfloat16, device_map="auto")
processor = AutoProcessor.from_pretrained(MODEL_ID)

In [ ]:
# @title 質問用の関数を用意する
import json
import re

import av
import numpy as np
from PIL import Image, ImageDraw, ImageOps
from transformers.video_utils import VideoMetadata

VIDEO_EXT = (".mp4", ".mov", ".webm", ".avi", ".mkv")


def load_image(path, max_side=1280):
    """画像を RGB で読む。大きすぎる画像はトークンが増えて遅くなるので縮める (座標は 0〜1000 の比率なので影響しない)"""
    img = path if isinstance(path, Image.Image) else ImageOps.exif_transpose(Image.open(path))
    img = img.convert("RGB")
    img.thumbnail((max_side, max_side))
    return img


def read_video(path, fps=4, max_side=640, max_frames=256):
    """動画を fps 枚/秒に間引いて読む。返すのはコマの配列と、各コマの時刻を伝えるための情報 (VideoMetadata)。
    Reasoner は各コマに「<1.5 seconds>」のような時刻を付けて読むので、時刻を答える問題 (第4章) にはこの情報が欠かせない"""
    frames, t_next = [], 0.0
    with av.open(path) as c:
        for f in c.decode(video=0):
            if f.time is None or f.time + 1e-6 < t_next:
                continue
            img = f.to_image()
            img.thumbnail((max_side, max_side))  # 1 コマのトークン数を抑える
            frames.append(np.asarray(img.convert("RGB")))
            t_next += 1 / fps
            if len(frames) >= max_frames:
                break
    n = len(frames)
    return np.stack(frames), VideoMetadata(total_num_frames=n, fps=fps, frames_indices=list(range(n)), duration=n / fps)


def ask(prompt, media=None, thinking=False, max_new_tokens=2048, fps=4, seed=0):
    """画像・動画 (なくてもよい) と質問を渡して、答えの文字列を返す。

    thinking=True のときは考えた過程 (<think>…</think>) を出してから答える。考えた過程は ask.last_think に残る。
    動画は fps 枚/秒で間引いて読む (推奨は 4)。
    """
    media = [] if media is None else media if isinstance(media, (list, tuple)) else [media]
    content, metadata = [], []
    for m in media:  # 画像・動画は質問文より前に置く (公式の推奨)
        if isinstance(m, str) and m.lower().endswith(VIDEO_EXT):
            frames, meta = read_video(m, fps=fps)
            content.append({"type": "video", "video": frames})
            metadata.append(meta)
        else:
            content.append({"type": "image", "image": load_image(m)})
    content.append({"type": "text", "text": prompt})
    # 動画は read_video で間引き済みなので、ここではもう間引かない
    video_kwargs = {"video_metadata": metadata, "do_sample_frames": False} if metadata else {}
    inputs = processor.apply_chat_template(
        [{"role": "user", "content": content}],
        tokenize=True, add_generation_prompt=True, return_dict=True, return_tensors="pt",
        enable_thinking=thinking, **video_kwargs,
    ).to(model.device)
    # 公式のプロンプトガイドのサンプリング設定 (考える / 考えない で値が違う)
    sampling = dict(temperature=0.6, top_p=0.95, top_k=20) if thinking else dict(temperature=0.7, top_p=0.8, top_k=20)
    torch.manual_seed(seed)
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=True, **sampling)
    text = processor.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    ask.last_think = ""
    if "</think>" in text:
        think, text = text.split("</think>", 1)
        ask.last_think = think.replace("<think>", "").strip()
    return text.strip()


def parse_json(text):
    """答えの中の JSON (```json … ``` で囲まれていてもよい) を取り出す。読めなければ None。
    小さなモデルの答えは、最後に余計な括弧が付く・真偽値の後ろに " が付く・最後に余計なカンマがある、
    といった崩れ方をするので、括弧の対応を数えて最初のまとまりだけを取り出し、よくある崩れを直してから読む"""
    m = re.search(r"```(?:json)?\s*(.*?)```", text, re.S)
    body = m.group(1) if m else text
    starts = [i for i in (body.find("["), body.find("{")) if i >= 0]
    if not starts:
        return None
    start = min(starts)
    depth, in_str, esc, end = 0, False, False, len(body)
    for k in range(start, len(body)):
        ch = body[k]
        if in_str:
            esc = (ch == "\\") and not esc
            if ch == '"' and not esc:
                in_str = False
            continue
        if ch == '"':
            in_str = True
        elif ch in "[{":
            depth += 1
        elif ch in "]}":
            depth -= 1
            if depth == 0:
                end = k + 1
                break
    chunk = body[start:end]
    fixed = re.sub(r'\b(true|false|null)"', r"\1", re.sub(r",\s*([\]}])", r"\1", chunk))
    for attempt in (chunk, fixed):
        try:
            return json.loads(attempt)
        except json.JSONDecodeError:
            pass
    print("(JSON として読めませんでした)")
    return None


def load_reasoner():
    """Reasoner を読み込む (生成の章で、Generator と入れ替えて使うとき用)"""
    global model, processor
    from transformers import AutoModelForImageTextToText, AutoProcessor
    model = AutoModelForImageTextToText.from_pretrained("nvidia/Cosmos3-Edge", dtype=torch.bfloat16, device_map="auto")
    processor = AutoProcessor.from_pretrained("nvidia/Cosmos3-Edge")

## 1. 画像を細かく説明させる

公式のプロンプトガイドと同じ「Caption the image in detail.」を、ロボットの頭のカメラから撮った写真に使います。

In [ ]:
img = fetch(f"{FW}/reasoner/assets/robot_153.jpg")
display(IPImage(img, width=480))
print(ask("Caption the image in detail.", img))

## 2. 画像について質問する

質問は日本語でも通りますが、Edge (4B) は英語のほうが安定します。日本語で読みたいときは「Answer in Japanese.」と付けるのが手軽です。

In [ ]:
img = fetch(f"{HF}/example_reasoning_input.png")
display(IPImage(img, width=480))
print(ask("What objects are on the table? List each object with its color.", img))
print("---")
print(ask("テーブルの上にある物を、色といっしょに挙げてください。Answer in Japanese.", img))

## 3. 動画を説明させる

動画は `fps` 枚/秒に間引いてから読みます。公式の推奨は 4 fps です。長い動画ほどトークンが増えて遅くなるので、まずは数秒〜十数秒の動画で試してください。

In [ ]:
video = fetch(f"{FW}/reasoner/assets/video_caption.mp4")
display(Video(video, embed=True, width=480))
print(ask("Describe the video in detail.", video))

## 4. 考えるモードで推論させる

`thinking=True` にすると、答える前に `<think>` … `</think>` の中で考えてから答えます。見えていることだけでは決まらない質問 (重さ・強度・因果など) で効きます。
時間はかかるので、見たままを答えれば済む質問では `thinking=False` (既定) にします。

In [ ]:
video = fetch(f"{FW}/reasoner/assets/common_sense_reasoning.mp4")
display(Video(video, embed=True, width=480))
answer = ask("Can the countertop support the weight of the juicers?", video, thinking=True)
print("--- 考えた過程 ---\n", ask.last_think[:2000])
print("\n--- 答え ---\n", answer)

## 5. 今の状況と、次に起きることを予想させる

世界モデルとしての Reasoner の見どころです。映っている人やロボットが何をしていて、次に何をしそうかを聞きます。

In [ ]:
video = fetch(f"{FW}/reasoner/assets/situation_understanding.mp4")
display(Video(video, embed=True, width=480))
print(ask("What is the person doing with the skillet? What will the person likely do next in this situation?", video))

## 6. 見えているものを JSON で出させる

ロボットやアプリから使うときは、文章よりも決まった形の JSON のほうが扱いやすくなります。
欲しいキーと例を質問文に書くと、その形で返してきます。`parse_json` で Python の値に戻せます。

In [ ]:
img = fetch(f"{HF}/example_reasoning_input.png")
answer = ask("""List every object on the table.
Return a JSON list. Each item has "name", "color", "position" (one of "left", "center", "right")
and "graspable" (true if a robot gripper could pick it up).
Example: [{"name": "cup", "color": "white", "position": "left", "graspable": true}]""", img)
print(answer)
objects = parse_json(answer)
if objects:
    for o in objects:
        print(f"{o.get('name')}: {o.get('color')}, {o.get('position')}, つかめる={o.get('graspable')}")

## 7. 自分の画像・動画で試す

下のセルを実行するとアップロード欄が出ます。画像でも動画 (mp4) でも構いません。
`question` を書き換えて実行してください。

In [ ]:
# @title 自分のファイルで試す (colab-only)
from google.colab import files

question = "Describe this scene in detail. What is likely to happen next?"  # @param {type:"string"}
thinking = False  # @param {type:"boolean"}

up = files.upload()
path = next(iter(up))
if path.lower().endswith(VIDEO_EXT):
    display(Video(path, embed=True, width=480))
else:
    display(IPImage(path, width=480))
answer = ask(question, path, thinking=thinking)
if thinking:
    print("--- 考えた過程 ---\n", ask.last_think[:2000], "\n")
print(answer)

## うまくいかないとき

- **答えが毎回少し違う**: サンプリングしているためです。`ask(..., seed=1)` のように seed を変えると別の答えになります。同じ seed なら同じ答えになります
- **日本語が不自然**: 英語で聞いて英語で答えさせ、必要なら翻訳するのが確実です
- **動画で GPU メモリが足りない**: `ask(..., fps=2)` にして読むコマを減らすか、動画を短く切ってください
- **考えるモードが途中で切れる**: `max_new_tokens=4096` に増やしてください